<font color='green'>***Installation and Libraries Import***</font>
---
--- 

In [ ]:
# Source - https://stackoverflow.com/a
# Posted by Nielsou Akbrg
# Retrieved 2026-01-27, License - CC BY-SA 4.0

%pip install torch torchvision matplotlib
import os,sys,importlib,llm,benchmark,time,ray,pickle,copy,csv,math,json,pandas, sklearn,torch,random,torch,glob
import pickle
import seaborn as sns
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, confusion_matrix
import flwr as fl
from collections import OrderedDict,Counter
from pprint import pprint
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import ConcatDataset, DataLoader

print("All modules loaded successfully!")
print("FLWR version:", fl.__version__)
print("Ray version:", ray.__version__)
print("Torch version:", torch.__version__)

libraries_to_uninstall = [
    "tb-nightly==2.18.0a20240701",
    "tensorboard==2.16.2",
    "tensorboard-data-server==0.7.2",
    "tensorboard-plugin-wit==1.8.1",
    "tensorflow==2.16.2",
    "tensorflow-io-gcs-filesystem==0.37.0",
    "termcolor==2.4.0",
    "terminado==0.18.1",
    "tf-estimator-nightly==2.8.0.dev2021122109",
    "tf_keras-nightly==2.18.0.dev2024070109",
    "tf-nightly==2.18.0.dev20240626"
]
for library in libraries_to_uninstall:
    os.system(f"pip uninstall -y {library}")
print("All modules loaded successfully!")
print("FLWR version:", fl.__version__)
print("Ray version:", ray.__version__)
print("Torch version:", torch.__version__)

In [ ]:

from typing import Dict, List, Optional, Tuple
import numpy as np
import pandas as pd
import torch.nn.functional as F
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split, Subset, Dataset, WeightedRandomSampler
from flwr.common import Metrics
from sklearn.preprocessing import MinMaxScaler, StandardScaler    
from sklearn.model_selection import train_test_split
print(fl.__version__)
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(DEVICE)

<font color='Brown'>***Constants***</font>
---
--- 

In [44]:

round_accuracy = {}
round_loss = {}
with open("config.json", "r") as f: #imports configuration constants from config.json 
    CFG = json.load(f) # requires config.json to be in the same directory 
NUM_CLIENTS = CFG["NUM_CLIENTS"]
ROUNDS = CFG["ROUNDS"]
BATCH_SIZE = CFG["BATCH_SIZE"]
LEARNING_RATE = CFG["LEARNING_RATE"]
EPOCHS = CFG["EPOCHS"]
DATA_GROUPS = CFG["DATA_GROUPS"]
BATCH_ROUND = CFG["BATCH_ROUND"]
NUM_ATCKS = CFG["NUM_ATCKS"]
FL = CFG["FL"]
EMB_DIM= CFG["EMB_DIM"]
MLP_DIM= CFG["MLP_DIM"]
MODE = CFG["MODE"]
INPUT_SIZE = CFG["INPUT_SIZE"]
HIDDEN1_SIZE = CFG["HIDDEN1_SIZE"]
HIDDEN2_SIZE = CFG["HIDDEN2_SIZE"]
TRAIN= CFG["TRAIN"]
G = CFG["G"]
LLM=CFG["LLM"]
LLM_TRIGGER_ROUND = CFG["LLM_TRIGGER_ROUND"]
DATA_DIR = CFG["DATA_DIR"]
SIZE_ROUND = BATCH_ROUND * BATCH_SIZE * NUM_CLIENTS
PATH = CFG["PATH_TEMPLATE"].format(MODE=MODE,FL=FL,NUM_CLIENTS=NUM_CLIENTS,NUM_ATCKS=NUM_ATCKS,ROUNDS=ROUNDS,EPOCHS=EPOCHS,LEARNING_RATE=LEARNING_RATE,
DATA_GROUPS=DATA_GROUPS,LLM=LLM
)
os.makedirs(PATH, exist_ok=True)

<font color='Light Blue'>***Dataset Preparations***</font>
---
--- 

In [ ]:
TrafficData = {}  # Root dictionary for traffic analysis
TrafficData['Dataset'] = {}  # Nested dictionary to hold individual DataFrames
sets_names = ['30', '100', '70', '50', 'testing']  # List of subset suffixes to load
for DATA_NUM in sets_names:
    # Build file path and load CSV; skip malformed lines and ignore quoting characters
    TrafficData['Dataset'][DATA_NUM] = pd.read_csv(os.path.join(DATA_DIR, f"2_Dataset_{NUM_ATCKS}_Attack_{DATA_NUM}_normal.csv"), low_memory=False, quoting=csv.QUOTE_NONE, on_bad_lines="skip")
    print(DATA_NUM, TrafficData['Dataset'][DATA_NUM].shape)  # Print subset name and row/column count
for DATA_NUM in TrafficData['Dataset']:
    # Shuffle all rows randomly using a fixed seed and reset the index to maintain clean ordering
    TrafficData['Dataset'][DATA_NUM] = TrafficData['Dataset'][DATA_NUM].sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
TrafficData['Split'] = {}  # Dictionary to store data split into smaller chunks
sets_training = ['30', '100', '70', '50']  # Keys for subsets that require splitting
for DATA_NUM in sets_training:
    TrafficData['Split'][DATA_NUM] = np.array_split(TrafficData['Dataset'][DATA_NUM], DATA_GROUPS)  # Divide each DataFrame into DATA_GROUPS number of nearly equal-sized arrays
# Initialize the combined DataFrame with the first chunk (index 0) from all training sets
TrafficData['Combined'] = pd.concat([TrafficData['Split']['30'][0], TrafficData['Split']['100'][0], TrafficData['Split']['70'][0], TrafficData['Split']['50'][0]]).reset_index(drop=True)
for GROUP in range(1, DATA_GROUPS):
    # Iteratively append subsequent chunks to the combined DataFrame and reset indices
    TrafficData['Combined'] = pd.concat([TrafficData['Combined'], TrafficData['Split']['30'][GROUP], TrafficData['Split']['100'][GROUP], TrafficData['Split']['70'][GROUP], TrafficData['Split']['50'][GROUP]]).reset_index(drop=True)
print(TrafficData['Combined'].shape)  # Verify final dimensions of the merged dataset

In [ ]:
TrafficData['Train'] = {}  # Initialize dictionary for training data components
TrafficData['Train']['X'] = TrafficData['Combined'].iloc[:, 0:-1]  # Extract all columns except the last as features
TrafficData['Train']['y'] = TrafficData['Combined'].iloc[:, -1]  # Extract only the last column as the target label
print(TrafficData['Train']['X'].shape)  # Verify the feature matrix dimensions
print(TrafficData['Train']['y'].shape)  # Verify the label vector dimensions
TrafficData['Test'] = {}  # Initialize dictionary for evaluation/test data components
TrafficData['Test']['X'] = TrafficData['Dataset']['testing'].iloc[:, 0:-1]  # Isolate testing features
TrafficData['Test']['y'] = TrafficData['Dataset']['testing'].iloc[:, -1]  # Isolate testing labels
print("Last column name (Label):", TrafficData['Combined'].columns[-1])  # Confirm the target column identity
print("First 5 values in y:", TrafficData['Train']['y'][:5])  # Sanity check for label formatting 

In [ ]:
# Choose normalization method: Standard (mean=0, std=1) for 'TT' mode or MinMax (range 0-1) otherwise
scaler = StandardScaler() if MODE == "TT" else MinMaxScaler() # Compute the necessary parameters (mean/std or min/max) from the training features only
model = scaler.fit(TrafficData['Train']['X'])# Serialize and save the fitted scaler to a file for future inference or consistency
with open(f"{PATH}/scaler.pkl", "wb") as f:
    pickle.dump(model, f) # Apply the scaling transformation to the training feature set
TrafficData['Train']['X'] = model.transform(TrafficData['Train']['X']) # Apply the same scaling parameters to the test set to avoid data leakage
TrafficData['Test']['X'] = model.transform(TrafficData['Test']['X'])# Convert training features and labels from DataFrames/Series to NumPy arrays for model compatibility
TrafficData['Train']['X'], TrafficData['Train']['y'] = np.array(TrafficData['Train']['X']), np.array(TrafficData['Train']['y'])
print(type(TrafficData['Train']['X']), type(TrafficData['Train']['y'])) # Confirm data types are now numpy.ndarray
print(TrafficData['Train']['X'].shape, TrafficData['Train']['y'].shape) # Verify feature and label dimensions match
# Convert test features and labels to NumPy arrays for final evaluation
TrafficData['Test']['X'], TrafficData['Test']['y'] = np.array(TrafficData['Test']['X']), np.array(TrafficData['Test']['y'])
print(type(TrafficData['Test']['X']), type(TrafficData['Test']['y'])) # Confirm test set data types
print(TrafficData['Test']['X'].shape, TrafficData['Test']['y'].shape) # Verify test set dimensions

In [ ]:
TrafficData['ROUNDS'] = {}  # Dictionary to store data partitions for each training round
for ROUND in range(1, ROUNDS + 1):
    TrafficData['ROUNDS'][ROUND] = {}  # Initialize a sub-dictionary for each specific round
SIZE_Demo = SIZE_ROUND  # Set the initial upper bound for the data slice
for ROUND in range(1, ROUNDS + 1):
    if ROUND == 1:# Extract the first chunk of data for Round 1 using the initial window size     
        TrafficData['ROUNDS'][ROUND]['X'] = TrafficData['Train']['X'][:SIZE_Demo]
        TrafficData['ROUNDS'][ROUND]['y'] = TrafficData['Train']['y'][:SIZE_Demo]
    else:
        # Define the slice range based on the previous end point and the fixed round size
        print((SIZE_Demo - SIZE_ROUND), SIZE_Demo)  # Debugging: show current slice indices
        TrafficData['ROUNDS'][ROUND]['X'] = TrafficData['Train']['X'][(SIZE_Demo - SIZE_ROUND):SIZE_Demo]
        TrafficData['ROUNDS'][ROUND]['y'] = TrafficData['Train']['y'][(SIZE_Demo - SIZE_ROUND):SIZE_Demo]
    SIZE_Demo = SIZE_Demo + SIZE_ROUND  # Increment the pointer to the next data block
for ROUND in TrafficData['ROUNDS']:
    print("ROUND: ", ROUND, TrafficData['ROUNDS'][ROUND]['X'].shape, TrafficData['ROUNDS'][ROUND]['y'].shape) # Display the allocated feature and label shapes for each training round to verify distribution
print(SIZE_Demo, SIZE_ROUND)  # Print the final pointer position and the step size

In [50]:
class ClassifierDataset(Dataset):
    def __init__(self, X_data, y_data):
        self.X_data = X_data
        self.y_data = y_data
    def __getitem__(self, index):
        return self.X_data[index], self.y_data[index]
    def __len__ (self):
        return len(self.X_data)

In [51]:
TrafficData['trainsets']={}
for ROUND in range(1, ROUNDS+1):
    TrafficData['trainsets'][ROUND]= ClassifierDataset(torch.from_numpy(TrafficData['ROUNDS'][ROUND]['X']).float(), torch.from_numpy(TrafficData['ROUNDS'][ROUND]['y']).long())
TrafficData['testset'] = ClassifierDataset(torch.from_numpy(TrafficData['Test']['X']).float(), torch.from_numpy(TrafficData['Test']['y']).long())

In [52]:
def load_train(numberofclients, ROUND):    
    # Use the actual dataset length to compute balanced portions per client
    dataset_len = len(TrafficData['trainsets'][ROUND])
    if dataset_len == 0:
        # Return an empty list of loaders to avoid indexing errors downstream
        return [DataLoader(Subset(TrafficData['trainsets'][ROUND], []), batch_size=BATCH_SIZE, shuffle=False) for _ in range(numberofclients)]
    # Distribute samples as evenly as possible across clients (handle remainders)
    num_portions = int(numberofclients)
    base_portion = dataset_len // num_portions
    remainder = dataset_len % num_portions
    portion_indices = []
    start_idx = 0
    for i in range(num_portions):
        # First `remainder` clients receive one extra sample
        sz = base_portion + (1 if i < remainder else 0)
        end_idx = start_idx + sz
        if sz > 0:
            portion_indices.append(list(range(start_idx, end_idx)))
        else:
            portion_indices.append([])
        start_idx = end_idx 
    # Build Subset and DataLoader for each client (safe: indices are within [0, dataset_len))
    portion_datasets = [Subset(TrafficData['trainsets'][ROUND], indices) for indices in portion_indices]
    portion_loaders = [DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False) for dataset in portion_datasets]            
    return portion_loaders
def load_test(numberofclients):    
    testloader = DataLoader(TrafficData['testset'], batch_size=BATCH_SIZE, shuffle=False)
    return testloader

In [53]:
Dataloaders = {}
for ROUND in range(1, ROUNDS+1):
    Dataloaders[ROUND] = load_train(NUM_CLIENTS, ROUND)
Dataloaders['Test'] = load_test(NUM_CLIENTS)

In [ ]:
# Diagnostic: print per-client dataset sizes for the first few rounds to verify splits
for ROUND in range(1, min(6, ROUNDS+1)):
    loaders = Dataloaders.get(ROUND, None)
    if loaders is None:
        print(f"Dataloaders for Round {ROUND} not found (run the rebuild cell).")
        continue
    sizes = [len(loader.dataset) for loader in loaders]
    print(f"Round {ROUND}: per-client sizes (first 12): {sizes[:12]}")
    print(f"  Total samples this round: {sum(sizes)}\n")

In [ ]:
for i, batch in enumerate(Dataloaders['Test']):
    batch_size = len(batch[0])  # Assuming the first element of the batch is the data
    print(f"Batch {i+1} size: {batch_size}")
    if batch_size != 64:
        print(f"Batch {i+1} does not contain 64 records.")
        break

In [56]:
def build_llm_results_snapshot(acc_list, loss_list):
    """
    Build the minimal structure lam.py expects.
    """
    return {
        "Accuracy": acc_list.copy(),
        "Loss": loss_list.copy(),
        "Precision": [],
        "Recall": [],
        "F1_Score": [],
    }
def build_llm_fl_results_snapshot(global_acc, client_accs):
    """
    Build the exact structure lam.py expects for FL.
    
    global_acc: List[float]  (per-round global accuracy)
    client_accs: List[List[float]]  (per-round per-client accuracy)
    """
    import numpy as np

    mean_acc = [float(np.mean(c)) for c in client_accs]
    std_acc  = [float(np.std(c)) for c in client_accs]
    min_acc  = [float(np.min(c)) for c in client_accs]
    max_acc  = [float(np.max(c)) for c in client_accs]

    return {
        "global_accuracy": global_acc,
        "mean_client_accuracy": mean_acc,
        "std_client_accuracy": std_acc,
        "min_client_accuracy": min_acc,
        "max_client_accuracy": max_acc,
    }


In [ ]:
for i, batch in enumerate(Dataloaders[5][0]):
    batch_size = len(batch[0])  # Assuming the first element of the batch is the data
    print(f"Batch {i+1} size: {batch_size}")
    if batch_size != 64:
        print(f"Batch {i+1} does not contain 64 records.")
        break

In [ ]:
from collections import 
for CLUSTER in range (1, 9):
    DEVICE_PERCENTAGE = []
    for DEVICE__ in range(0,NUM_CLIENTS):
        for i, batch in enumerate(Dataloaders[CLUSTER][DEVICE__]):
            _, labels = batch
            class_counts = Counter(labels.numpy())
            total_records = sum(class_counts.values())
            class_0_count = class_counts.get(0, 0)
            percentage_class_0 = (class_0_count / total_records) * 100
            DEVICE_PERCENTAGE.append(percentage_class_0)
   
    chunk_size = 6
    averages = [sum(DEVICE_PERCENTAGE[i:i + chunk_size]) / chunk_size for i in range(0, len(DEVICE_PERCENTAGE), chunk_size)]

    chunk_size_4 = 4
    averages = [sum(averages[i:i + chunk_size_4]) / chunk_size_4 for i in range(0, len(averages), chunk_size_4)]

    print(averages)

In [59]:
# del TrafficData

<font color='Red'>***Neural Network***</font>
---
--- 

In [60]:
if MODE == 'DNN':
    class Net(nn.Module):
        def __init__(self):
            super(Net, self).__init__()        
            self.layer_1 = nn.Linear(INPUT_SIZE, HIDDEN2_SIZE )
            self.layer_2 = nn.Linear(HIDDEN2_SIZE , HIDDEN1_SIZE )
            self.layer_out = nn.Linear(HIDDEN1_SIZE , NUM_ATCKS+1) 
            self.relu = nn.ReLU()
        def forward(self, x):
            x = self.layer_1(x)
            x = self.relu(x)
            x = self.layer_2(x)
            x = self.relu(x)
            x = self.layer_out(x)        
            return x

<font color='Yellow'>***Tab Transformer***</font>
---
--- 

In [61]:
if MODE == 'TT':
    class TabTransformer(nn.Module):
        def __init__(self, num_features, emb_dim, mlp_dim, num_layers=4, num_heads=4,  num_classes=NUM_ATCKS+1):
            super().__init__()
            self.num_features = num_features
            self.emb_dim = emb_dim
            self.feature_projection = nn.Linear(1, emb_dim)
            self.feature_id_emb = nn.Parameter(torch.randn(num_features, emb_dim))
            encoder_layer = nn.TransformerEncoderLayer(d_model=emb_dim, nhead=num_heads, dim_feedforward=mlp_dim, batch_first=True)
            self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
            self.output_head = nn.Sequential(
                nn.Linear(emb_dim, mlp_dim),
                nn.ReLU(),
                nn.Linear(mlp_dim, num_classes)
            )
        def forward(self, x):
            x = x.unsqueeze(-1)
            emb = self.feature_projection(x) 
            emb = emb + self.feature_id_emb.unsqueeze(0) 
            z = self.transformer(emb) 
            pooled = z.mean(dim=1) 
            logits = self.output_head(pooled) 
            return logits




<font color='Purple'>***Model Selection***</font>
---
--- 

In [62]:
num_features = TrafficData["Train"]["X"].shape[1]
def build_model(emb_dim=None, mlp_dim=None):
    if MODE == "DNN":
        return Net()
    elif MODE == "TT":
        emb_dim = emb_dim if emb_dim is not None else CFG["EMB_DIM"]
        mlp_dim = mlp_dim if mlp_dim is not None else CFG["MLP_DIM"]
        return TabTransformer(
            num_features,
            emb_dim = emb_dim,
            mlp_dim = mlp_dim
        )
    else:
        raise ValueError(f"Unknown MODE={MODE}")
        
def load_global_model(path):
    checkpoint = torch.load(path, map_location="cpu")
    # Old checkpoints fallback
    if "MODE" not in checkpoint:
        model = build_model()
        model.load_state_dict(checkpoint)
        return model  
    model = build_model()
    model.load_state_dict(checkpoint["state_dict"])
    model.eval()

    return model



In [ ]:
init_ckpt = f"0_Input_Random_Model_{MODE}.pth" # CREATE INITIAL GLOBAL MODEL (REQUIRED FOR FL)
model = build_model()
model.eval()
torch.save(model.state_dict(), init_ckpt)
print(f" Initial model saved: {init_ckpt}")


<font color='Orange'>***Train & Test Functions***</font>
---
--- 

In [64]:
def architecture_changed(old_cfg, new_cfg):
    if old_cfg["MODE"] == "DNN":
        return (
            old_cfg["HIDDEN1_SIZE"] != new_cfg["HIDDEN1_SIZE"] or
            old_cfg["HIDDEN2_SIZE"] != new_cfg["HIDDEN2_SIZE"]
        )
    return False

In [65]:
def train(model, trainloader, epochs: int, lr:float,verbose=True):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    model.train()
    prediction_matrix = []
    actual_matrix = []
    acc_matrix = []
    loss_matrix = []
    # Safeguard: if the trainloader has no samples, skip training
    try:
        dataset_len = len(trainloader.dataset)
    except Exception:
        dataset_len = 0
    if dataset_len == 0:
        if verbose:
            print("Warning: trainloader has 0 samples, skipping training.")
        return prediction_matrix, actual_matrix, acc_matrix, loss_matrix
    for epoch in range(epochs):
        correct, total, epoch_loss = 0, 0, 0.0
        for images, labels in trainloader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            # Accumulate loss as scalar sum over samples
            epoch_loss += loss.item() * labels.size(0)
            total += labels.size(0)
            correct += (torch.max(outputs.data, 1)[1] == labels).sum().item()
            predictions = torch.max(outputs.data, 1)[1]
            prediction_matrix.append(predictions.tolist())
            actual_matrix.append(labels.tolist())
        # Avoid division by zero if total==0 for safety
        if total == 0:
            epoch_loss_val = 0.0
            epoch_acc = 0.0
        else:
            epoch_loss_val = epoch_loss / total
            epoch_acc = correct / total
        loss_matrix.append(epoch_loss_val)
        acc_matrix.append(epoch_acc)
        if verbose:
            print(f"Epoch {epoch+1}/{epochs} | Loss: {epoch_loss_val:.6f} | Acc: {epoch_acc:.4f}")
    return prediction_matrix, actual_matrix, acc_matrix, loss_matrix 
#modified it to make compatible with llm
def test(model, testloader):
    criterion = nn.CrossEntropyLoss()
    model.eval()
    model.to(DEVICE)  #  ENSURE MODEL IS ON DEVICE
    correct, total, loss_sum = 0, 0, 0.0
    prediction_matrix = []
    actual_matrix = []
    acc_matrix = []
    loss_matrix = []

    with torch.no_grad():
        for images, labels in testloader:
            images = images.to(DEVICE)   #MOVE DATA
            labels = labels.to(DEVICE)

            outputs = model(images)
            loss_sum += criterion(outputs, labels).item() * labels.size(0)

            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            prediction_matrix.extend(predicted.cpu().tolist())
            actual_matrix.extend(labels.cpu().tolist())

    avg_loss = loss_sum / total if total > 0 else 0.0
    accuracy = correct / total if total > 0 else 0.0

    acc_matrix.append(accuracy)
    loss_matrix.append(avg_loss)

    print(f"Evaluation: loss={avg_loss:.6f}, acc={accuracy:.4f}")
    return avg_loss, accuracy, prediction_matrix, actual_matrix, acc_matrix, loss_matrix


In [66]:
def net2wider_load(target_model, source_state_dict, noise_std=1e-5):
    """
    Implements Net2WiderNet logic with a status report return.
    
    Returns:
        copied_exact: List of keys where shapes matched exactly.
        copied_partial: List of keys where Net2Wider transformation was applied.
        skipped: List of keys that were ignored due to missing keys or incompatible dims.
    """
    tgt_state = target_model.state_dict()
    copied_exact, copied_partial, skipped = [], [], []

    for k, v in source_state_dict.items():
        if k not in tgt_state:
            skipped.append(k)
            continue
            
        tgt_v = tgt_state[k]
        
        # Case 1: Exact Match
        if tgt_v.shape == v.shape:
            tgt_state[k] = v.clone()
            copied_exact.append(k)
            continue

        # Case 2: Net2Wider Transformation (Partial/Expanded Copy)
        try:
            # Handle Linear Layer Weights (2D)
            if v.ndim == 2 and tgt_v.ndim == 2:
                o_new, i_new = tgt_v.shape
                o_old, i_old = v.shape
                
                # Create replication maps (Modulo logic)
                row_map = torch.arange(o_new) % o_old
                col_map = torch.arange(i_new) % i_old
                
                # Step 1: Replicate existing weights
                new_w = v[row_map][:, col_map]
                
                # Step 2: Scale outgoing weights (Function Preservation)
                # Count how many times each input index was replicated
                counts = torch.bincount(col_map, minlength=i_old).float()
                scaling = 1.0 / counts[col_map]
                new_w = new_w * scaling.view(1, -1)
                
                
                # Step 3: Symmetry Breaking (Tiny noise on new slots only)
                if noise_std > 0:
                    noise = torch.randn_like(new_w) * noise_std
                    mask = torch.ones_like(new_w)
                    mask[:o_old, :i_old] = 0 # Protect the original weights
                    new_w = new_w + (noise * mask)
                
                tgt_state[k] = new_w
                copied_partial.append(k)

            # Handle Biases or 1D Vectors
            elif v.ndim == 1 and tgt_v.ndim == 1:
                l_new, l_old = tgt_v.shape[0], v.shape[0]
                idx_map = torch.arange(l_new) % l_old
                tgt_state[k] = v[idx_map].clone()
                copied_partial.append(k)
            
            else:
                skipped.append(k)

        except Exception:
            skipped.append(k)

    # Apply the transformed state to the model
    target_model.load_state_dict(tgt_state)
    
    return copied_exact, copied_partial, skipped

***Centralized DNN/TabTransformer Rounds***
---

In [67]:
# Centralized Training Loop DNN

import importlib
import benchmark
import llm

importlib.reload(benchmark)
importlib.reload(llm)

from benchmark import BenchmarkLogger
from llm import llm_mid_training_update

if not FL and TRAIN:
    def train_one_round(model, trainloader, epochs,lr):
        """Runs EPOCHS of training on this round's dataset."""
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=lr)
        model.train()

        for epoch in range(epochs):
            correct, total, running_loss = 0, 0, 0.0
            for Xbatch, ybatch in trainloader:
                Xbatch, ybatch = Xbatch.to(DEVICE), ybatch.to(DEVICE)

                optimizer.zero_grad()
                outputs = model(Xbatch)
                loss = criterion(outputs, ybatch)
                loss.backward()
                optimizer.step()

                running_loss += loss.item()
                total += ybatch.size(0)
                correct += (torch.max(outputs, 1)[1] == ybatch).sum().item()

            print(f"  Epoch {epoch+1}/{epochs} | "
                f"Loss: {running_loss/len(trainloader):.6f} | "
                f"Acc: {correct/total:.4f}")



    # ============================================================
    # Centralized Training Loop
    # ============================================================
    experiment_start_time = time.time()
    llm_trigger_time = None
    bench = BenchmarkLogger(
    output_path=PATH,
    mode="centralized",
    model_type=MODE
    )
    os.makedirs(PATH, exist_ok=True)
    print(" Starting centralized Round-by-Round training...\n")

    model = build_model().to(DEVICE)

    for Round in range(1, ROUNDS + 1):
        bench.start_round(Round)

        print("========================")
        print(f"   ROUND {Round} START")
        print("========================")

        # Build round dataset from client partitions
        client_datasets = [
            loader.dataset for loader in Dataloaders[Round]
            if hasattr(loader, "dataset") and len(loader.dataset) > 0
        ]

        round_dataset = ConcatDataset(client_datasets)
        trainloader = DataLoader(round_dataset, batch_size=BATCH_SIZE, shuffle=True)

        print(f"  Train samples this round: {len(round_dataset)}")

        # Train
        t0 = time.time()
        train_one_round(model, trainloader, EPOCHS,LEARNING_RATE)
        train_time = time.time() - t0


        # Evaluate (optional)
        print(f"  Evaluating round {Round} model...")
        t0 = time.time()
        avg_loss, avg_acc, *_ = test(model, Dataloaders["Test"])
        eval_time = time.time() - t0


        round_accuracy[Round] = [avg_acc]
        round_loss[Round] = [avg_loss]
        samples_this_round = len(round_dataset)

        bench.log_memory()

        bench.end_round(
        train_time=train_time,
        eval_time=eval_time,
        accuracy=avg_acc,
        loss=avg_loss,
        samples_this_round=samples_this_round,
        learning_rate=LEARNING_RATE,
        epochs=EPOCHS,
    )


        # Save model
        save_path = f"{PATH}/GlobalModel_{Round}.pth"
        torch.save(model.state_dict(), save_path)
        print(f" Saved: {save_path}")

        print("========================")
        print(f"   ROUND {Round} DONE")
        print("========================\n")
        if Round == LLM_TRIGGER_ROUND and LLM==True:
            llm_trigger_time = time.time()


            print("\nTriggering LLM mid-training update...")

            results_snapshot = build_llm_results_snapshot(
                acc_list=round_accuracy.get(Round, []),
                loss_list=round_loss.get(Round, []),

            )

            new_config = llm_mid_training_update(
                model_path=save_path,
                config_path="config.json",
                results_snapshot=results_snapshot,
                round_idx=Round,
                trigger_round=LLM_TRIGGER_ROUND,
                allow_architecture_change=True,
            )

            if new_config:
                old_cfg = CFG.copy()
                CFG.update(new_config)

                LEARNING_RATE = CFG["LEARNING_RATE"]  #  APPLIED
                EPOCHS = CFG["EPOCHS"]                #  APPLIED
                ROUNDS = CFG["ROUNDS"]                #  APPLIED

                print(
                    f"Updated LR={LEARNING_RATE}, "
                    f"local_epochs={EPOCHS}, "
                    f"total_rounds={ROUNDS}"
                )
                arch_changed = architecture_changed(old_cfg, CFG)

                if arch_changed:
                    print("Applying Net2Net architecture transfer")

                    old_model = model

                    # Build NEW architecture using updated CFG
                    widened_model = build_model().to(DEVICE)
                    if MODE == "DNN":
                        copied_exact, copied_partial, skipped = net2wider_load(
                            target_model=widened_model,
                            source_state_dict=old_model.state_dict(),
                            noise_std=1e-5,
                        )
                    else:
                        copied_exact, copied_partial, skipped = net2wider_load_tt(
                            target_model=widened_model,
                            source_state_dict=old_model.state_dict(),
                            noise_std=1e-5,
                        )

                    print(f"Net2Net summary:")
                    print(f"  Exact copied: {len(copied_exact)}")
                    print(f"  Expanded: {len(copied_partial)}")
                    print(f"  Skipped: {len(skipped)}")

                    # CRITICAL: replace the global model
                    model = widened_model

                    torch.save(
                        widened_model.state_dict(),
                        f"{PATH}/GlobalModel_{Round}_net2net.pth"
                    )

                print("LLM update applied — continuing training\n")
    total_time = time.time() - experiment_start_time
    bench.log_experiment_time(total_time)

    if llm_trigger_time:
        bench.log_llm_overhead(
            total_time - (llm_trigger_time - experiment_start_time))

    print("Centralized Round-by-Round training completed.")


<font color='Brown'>***Federated Learning***</font>
---

In [68]:
prediction_dict= {}
actual_dict= {}
accuracy_dict= {}
loss_dict= {}




In [69]:
if FL and TRAIN:
    def get_parameters(model) -> List[np.ndarray]:
        return [val.cpu().numpy() for _, val in model.state_dict().items()]
    def set_parameters(model, parameters: List[np.ndarray]):
        params_dict = zip(model.state_dict().keys(), parameters)
        state_dict = OrderedDict({k: torch.Tensor(v) for k, v in params_dict})
        model.load_state_dict(state_dict, strict=True)

In [70]:
if FL and TRAIN:        
    class FlowerClient(fl.client.NumPyClient):
        def __init__(self, cid, model, trainloader, FL_Update):
            self.cid = cid
            self.model = model
            self.trainloader = trainloader
            self.FL_Update = FL_Update

        def get_parameters(self, config):
            print(f"[Client {self.cid}] get_parameters")
            return get_parameters(self.model)

        def fit(self, parameters, config):
            import traceback
            try:
                local_epochs = config.get("local_epochs", EPOCHS)
                lr = config.get("learning_rate", LEARNING_RATE)
                print(f"[Client {self.cid}, round {self.FL_Update}] fit, config: {config}")

                # Set parameters sent by server
                set_parameters(self.model, parameters)

                # Train locally
                _1, _2, _3, _4 = train(self.model, self.trainloader, epochs=local_epochs,lr=lr)
                prediction_dict[f'C{self.cid}R{self.FL_Update}'] = _1
                actual_dict[f'C{self.cid}R{self.FL_Update}'] = _2
                accuracy_dict[f'C{self.cid}R{self.FL_Update}'] = _3
                loss_dict[f'C{self.cid}R{self.FL_Update}'] = _4

                # Determine number of examples contributed by this client (preferred: dataset length)
                num_examples = 0
                try:
                    num_examples = len(self.trainloader.dataset)
                except Exception:
                    # Fallback: sum batch sizes (non-consuming) if dataset missing
                    try:
                        num_examples = sum(batch[0].shape[0] for batch in self.trainloader)
                    except Exception:
                        num_examples = 0

                return get_parameters(self.model), int(num_examples), {}

            except Exception as e:
                tb = traceback.format_exc()
                print(f"[Client {self.cid}] fit exception: {e}\n{tb}")
                # Attempt to return current parameters so server can continue; return 0 examples
                try:
                    return get_parameters(self.model), 0, {}
                except Exception as e2:
                    print(f"[Client {self.cid}] Failed to return fallback parameters: {e2}")
                    # Let exception propagate if we cannot return safe fallback
                    raise

        def evaluate(self, parameters, config):
            print(f"[Client {self.cid}")
            print(f"[Client {self.cid}] evaluate, config: {config}")
            return None


In [71]:
if FL and TRAIN:
    def retrieve_and_sort_client_updates(global_model, round_number, client_ids):
        client_updates = {}
        for cid in client_ids:
            update_filename = f'EdgeCooperation/Performance/Results/C{cid}R{round_number}_update.pkl'
            with open(update_filename, 'rb') as update_file:
                client_update = pickle.load(update_file)
                client_updates[cid] = client_update
        client_contributions = {cid: calculate_weight_magnitude(global_model, update) for cid, update in client_updates.items()}
        sorted_clients = sorted(client_contributions.items(), key=lambda x: x[1], reverse=True)
        least_contributing_clients = sorted_clients[-3:]
        return sorted_clients, least_contributing_clients

    def calculate_weight_magnitude(global_model, client_update):
        """
        Calculate the L2 norm of the weight difference between the global model and client's updated model.
        
        Args:
        global_model (nn.Module): The global model before client update.
        client_update (list): List of numpy arrays representing client's updated model parameters.

        Returns:
        float: The L2 norm of the weight difference.
        """
        weight_diff = 0.0
        global_parameters = [param.detach().cpu().numpy() for param in global_model.parameters()]
        for global_param, client_param in zip(global_parameters, client_update):
            weight_diff += np.linalg.norm(global_param - client_param)
        return weight_diff


In [72]:
# Clients Function 
if FL and TRAIN:
    def General_Client():
        def client_fn(cid: int, Round: int) -> FlowerClient:
            clients_ids_list = TrainingListPerRound[Round]
            if int(cid) in clients_ids_list:
                model = build_model().to(DEVICE)
                trainloader = Dataloaders[Round][int(cid)]
                arg_ = Round
                return FlowerClient(cid, model, trainloader, arg_)
            else:
                raise ValueError(f"Client ID {cid} not found in the list for round {Round}")
        return client_fn

In [73]:
# Strategy Functions DNN
if FL and TRAIN:
    Global_Models = {}
    class SaveModelStrategy(fl.server.strategy.FedAvg):
        def __init__(self, additional_argument, *args, **kwargs):
            super().__init__(*args, **kwargs)
            self.additional_argument = additional_argument

        def aggregate_fit(self, rnd: int, results: List[Tuple[fl.server.client_proxy.ClientProxy, fl.common.FitRes]], failures: List[BaseException]) -> Optional[fl.common.NDArrays]:
            print(f"[SaveModelStrategy] aggregate_fit called for additional_argument={self.additional_argument}, rnd={rnd}, results={len(results)}, failures={len(failures)}")
            try:
                aggregated_parameters_tuple = super().aggregate_fit(rnd, results, failures)
                if not aggregated_parameters_tuple:
                    print(f"[SaveModelStrategy] No aggregated parameters returned for round {self.additional_argument}")
                    return aggregated_parameters_tuple

                aggregated_parameters, _ = aggregated_parameters_tuple
                if aggregated_parameters is None:
                    print(f"[SaveModelStrategy] aggregated_parameters is None for round {self.additional_argument}")
                    return aggregated_parameters_tuple

                # Convert `Parameters` to `List[np.ndarray]`
                aggregated_weights: List[np.ndarray] = fl.common.parameters_to_ndarrays(aggregated_parameters)

                # Convert `List[np.ndarray]` to PyTorch state_dict and save
                Global_Models[self.additional_argument] = build_model()
                params_dict = zip(Global_Models[self.additional_argument].state_dict().keys(), aggregated_weights)
                state_dict = OrderedDict({k: torch.tensor(v) for k, v in params_dict})
                try:
                    Global_Models[self.additional_argument].load_state_dict(state_dict, strict=True)
                except Exception as e:
                    print(f"[SaveModelStrategy] Failed loading state_dict into model for round {self.additional_argument}: {e}")

                try:
                    os.makedirs(PATH, exist_ok=True)
                    save_path = f"{PATH}/GlobalModel_{self.additional_argument}.pth"
                    torch.save({
                            "state_dict": Global_Models[self.additional_argument].state_dict(),
                            "MODE": MODE
                        }, save_path)

                    print(f"[SaveModelStrategy] Saved GlobalModel to {save_path}")
                except Exception as e:
                    print(f"[SaveModelStrategy] Failed to save GlobalModel_{self.additional_argument}: {e}")
                return aggregated_parameters_tuple
            except Exception as e:
                print(f"[SaveModelStrategy] Exception in aggregate_fit: {e}")
                raise
            


In [74]:
# Strategy Config 
if FL and TRAIN:
    def fit_config(server_round: int):
        return {
            "current_round": server_round,
            "local_epochs": CFG["EPOCHS"],
            "learning_rate": CFG["LEARNING_RATE"],
        }


In [75]:
# Federated Learning Training Loop 
import importlib,benchmark,llm
importlib.reload(benchmark)
importlib.reload(llm)
from benchmark import BenchmarkLogger
from llm import llm_mid_training_update
def get_parameters(model):
    return [val.detach().cpu().numpy() for val in model.state_dict().values()]

if FL and TRAIN:
    import time
    os.makedirs(PATH, exist_ok=True)

   
    print("Loading Initial Global Model")
    Global_Models = {}
    Global_Models[0] = build_model()

    init_ckpt = f"0_Input_Random_Model_{MODE}.pth"
    if not os.path.exists(init_ckpt):
        raise FileNotFoundError(f"Missing initial checkpoint: {init_ckpt}")

    Global_Models[0].load_state_dict(torch.load(init_ckpt, map_location=DEVICE))
    Global_Models[0].train()

   
    # Client participation list
   
    TrainingListPerRound = {
        rnd: list(range(NUM_CLIENTS))
        for rnd in range(1, ROUNDS + 1)
    }

    bench = BenchmarkLogger(
        output_path=PATH,
        mode="federated",
        model_type=MODE,
    )
    # Wall-clock timers
    experiment_start_time = time.time()
    llm_trigger_time = None
    # Dynamic FL loop
    Round = 1
    while Round <= ROUNDS:

        bench.start_round(Round)

        print("\n" + "=" * 60)
        print(f"Starting FL Round {Round}")
        print("=" * 60)

        prev_model = Global_Models.get(Round - 1, Global_Models[0])

        strategy = SaveModelStrategy(
            fraction_fit=1.0,
            fraction_evaluate=0.0,
            min_fit_clients=2,
            min_evaluate_clients=0,
            min_available_clients=2,
            on_fit_config_fn=fit_config,
            initial_parameters=fl.common.ndarrays_to_parameters(
                get_parameters(prev_model)
            ),
            additional_argument=Round,
        )

        client_factory = General_Client()   
        # FL round execution
        t0 = time.time()
        fl.simulation.start_simulation(
            client_fn=lambda cid, rnd=Round: client_factory(cid, rnd),
            num_clients=NUM_CLIENTS,
            config=fl.server.ServerConfig(num_rounds=1),
            client_resources={"num_cpus": 16, "num_gpus": 1},
            ray_init_args={"num_cpus": 16, "num_gpus": 1},
            strategy=strategy,
        )
        agg_time = time.time() - t0
        samples_this_round = sum(len(Dataloaders[Round][cid].dataset)for cid in range(NUM_CLIENTS))     
        # Load global model    
        model_path = f"{PATH}/GlobalModel_{Round}.pth"
        Global_Models[Round] = load_global_model(model_path).to(DEVICE)
        Global_Models[Round].train()
        # Evaluation
        t0 = time.time()
        avg_loss, avg_acc, *_ = test(Global_Models[Round], Dataloaders["Test"])
        eval_time = time.time() - t0

        bench.log_memory()

        bench.end_round(
        train_time=agg_time,
        eval_time=eval_time,
        accuracy=avg_acc,
        loss=avg_loss,
        samples_this_round=samples_this_round,
        learning_rate=CFG["LEARNING_RATE"],
        epochs=CFG["EPOCHS"],
    )
        # LLM MID-TRAINING UPDATE
        if Round == LLM_TRIGGER_ROUND and LLM==True:

            print("\nTriggering LLM mid-training update (FL)")
           

            tmp = build_model()
            

            llm_trigger_time = time.time()

            client_accs = [
                accuracy_dict[k][-1]
                for k in accuracy_dict
                if k.endswith(f"R{Round}") and accuracy_dict[k]
            ]

            fl_results_snapshot = build_llm_fl_results_snapshot(
                global_acc=[avg_acc],
                client_accs=client_accs,
            )
            new_config = llm_mid_training_update(
                model_path=model_path,
                config_path="config.json",
                results_snapshot=fl_results_snapshot,
                round_idx=Round,
                trigger_round=LLM_TRIGGER_ROUND,
                allow_architecture_change=True,
            )
            if new_config:
                old_cfg = CFG.copy()
                for k in ["LEARNING_RATE", "EPOCHS", "ROUNDS"]:
                    if k in new_config:
                        CFG[k] = new_config[k]
                LEARNING_RATE = CFG["LEARNING_RATE"]  
                EPOCHS = CFG["EPOCHS"]                
                ROUNDS = CFG["ROUNDS"] 
                  
                print(
                    f"Updated LR={LEARNING_RATE}, "
                    f"local_epochs={EPOCHS}, "
                    f"total_rounds={ROUNDS},"
                   

                )
                arch_changed = architecture_changed(old_cfg, CFG)

                if arch_changed:
                    print("Applying Net2Net architecture transfer")

                    old_model = Global_Models[Round]

                    # Build NEW architecture using updated CFG
                    widened_model = build_model().to(DEVICE)
                    if MODE == "DNN":
                        copied_exact, copied_partial, skipped = net2wider_load(
                            target_model=widened_model,
                            source_state_dict=old_model.state_dict(),
                            noise_std=1e-5,
                        )

                    # CRITICAL: replace the global model
                    Global_Models[Round] = widened_model

                    torch.save(
                        widened_model.state_dict(),
                        f"{PATH}/GlobalModel_{Round}_net2net.pth"
                    )

        print(f"End of FL Round {Round}")
        Round += 1

   
    # Experiment summary
   
    total_time = time.time() - experiment_start_time
    bench.log_experiment_time(total_time)

    if llm_trigger_time:
        bench.log_llm_overhead(
            total_time - (llm_trigger_time - experiment_start_time)
        )

    print("\n==========================================")
    print(" FEDERATED TRAINING COMPLETED")
    print(f" Total runtime: {total_time:.2f} seconds")
    print("==========================================")



<font color='Grey'>***Performance Testing***</font>
---
---


In [76]:
def write_round_summary_json(
    run_path: str,
    round_num: int,
    output_name: str = None,
    cooccur_rate_threshold: float = 0.10,  # include predicted->predicted pairs >= 10% of that "from" label
    top_k_pairs: int = 25,
    top_k_labels: int = 10
):
    """
    Write a predictions-only JSON summary for a given round.
    """
    # Label names must match the dataset's class encoding order
    LABEL_NAMES = [
        "Normal", "DDoS_UDP", "DDoS_ICMP", "DDoS_TCP", "DDoS_HTTP", "Password",
        "Vulnerability_scanner", "SQL_injection", "Uploading", "Backdoor", "Port_Scanning",
        "XSS", "Ransomware", "MITM", "OS_Fingerprinting"
    ]

    def load_pickle(filename):
        """
        Load a pickle file from the run directory.
        """
        full = os.path.join(run_path, filename)
        with open(full, "rb") as f:
            return pickle.load(f)
    pred = load_pickle(f"Global_{round_num}_pred")

    # Flatten batched predictions into a single sequence
    flat_pred = [int(x) for batch in pred for x in batch]

    def label_name(i: int) -> str:
        """
        Map a class index to a human-readable label.
        """
        return LABEL_NAMES[i] if 0 <= i < len(LABEL_NAMES) else f"class_{i}"

    n = len(flat_pred)
    if n == 0:
        raise ValueError(f"Round {round_num}: no samples found in predictions.")

    # Compute predicted label distribution
    pred_counts_raw = Counter(flat_pred)
    predicted_counts = {label_name(i): int(pred_counts_raw.get(i, 0)) for i in range(len(LABEL_NAMES))}
    predicted_percent = {k: (v / n) for k, v in predicted_counts.items()}
    inactive_labels = [k for k, v in predicted_counts.items() if v == 0]

    # Build predicted transition/co-occurrence pairs from consecutive predictions
    pair_counts_raw = Counter(zip(flat_pred[:-1], flat_pred[1:])) if n >= 2 else Counter()
    cooccur_pairs = []
    for (a, b), c in pair_counts_raw.items():
        if a == b:
            continue
        total_a = pred_counts_raw.get(a, 0)
        rate = (c / total_a) if total_a > 0 else 0.0
        if rate >= cooccur_rate_threshold:
            cooccur_pairs.append({
                "from_predicted": label_name(a),
                "to_predicted": label_name(b),
                "count": int(c),
                "rate": float(round(rate, 4))
            })

    cooccur_pairs.sort(key=lambda x: x["rate"], reverse=True)

    # Top-K lists for readability
    top_labels = [
        {"label": label_name(i), "count": int(cnt), "percent": float(round(cnt / n, 4))}
        for i, cnt in pred_counts_raw.most_common(top_k_labels)
    ]

    top_cooccur_by_count = [
        {"from_predicted": label_name(a), "to_predicted": label_name(b), "count": int(c)}
        for (a, b), c in pair_counts_raw.most_common(top_k_pairs)
        if a != b
    ]
    summary = {
        "round": int(round_num),
        "num_samples": int(n),
        "predicted_counts": predicted_counts,
        "predicted_percent": {k: float(round(v, 6)) for k, v in predicted_percent.items()},
        "inactive_labels": inactive_labels,
        "top_labels": top_labels,
        "cooccur_pairs": cooccur_pairs,
        "top_cooccur_by_count": top_cooccur_by_count,
        "cooccur_rate_threshold": float(cooccur_rate_threshold)
    }

    # Choose default output filename    
    if output_name is None:
        output_name = f"Round_{round_num}_Summary.json"
    # Write JSON to disk
    out_path = os.path.join(run_path, output_name)
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    print(f"[JSON] Saved {out_path}")

In [ ]:
import os
import glob

# Define the directory and file pattern
directory = PATH + '/'
pattern = "GlobalModel_*.pth"

# Find all matching files
files = glob.glob(os.path.join(directory, pattern))

# Extract numbers from file names
numbers = []
for file in files:
    base_name = os.path.basename(file)
    num_str = base_name.replace("GlobalModel_", "").replace(".pth", "")
    try:
        numbers.append(int(num_str))
    except ValueError:
        pass

# Determine the maximum number
max_num = max(numbers) if numbers else 0
print(max_num)

# Use the max_num in a loop
for num in range(1, max_num + 1):
    file_path = f"{PATH}/GlobalModel_{num}.pth"
    if os.path.exists(file_path):
        # Load the file or perform any operation you need
        print(f"Loading {file_path}")
    else:
        print(f"File {file_path} does not exist")

In [78]:
def view_pickle(path, G=None, suffix=None):
    print(G)
    if G is not None:
        rounds = [G]
    else:
        # Auto-detect how many Global_X files exist
        rounds = sorted({
            int(f.split("_")[1])
            for f in os.listdir(path)
            if f.startswith("Global_") and f.split("_")[1].isdigit()
        })

    suffixes = [suffix] if suffix else ["pred", "actual", "accurracy", "loss"]

    for g in rounds:
        for s in suffixes:
            filename = os.path.join(path, f"Global_{g}_{s}")
            if not os.path.exists(filename):
                print(f"Missing file: {filename}")
                continue
            try:
                with open(filename, "rb") as f:
                    data = pickle.load(f)
                print(f"File: {filename}")
                if isinstance(data, list) and len(data) > 2:
                    pprint(data[:2])
                    print(f"... ({len(data)} total items)")
                else:
                    pprint(data)
            except Exception as e:
                print(f"Error reading {filename}: {e}")



In [79]:
pred_test = {}
actual_test = {}
accuracy_test = {}
loss_test = {}

G = 0
criterion = nn.CrossEntropyLoss()

if TRAIN:

    for num in range(1, max_num + 1):

        print(f"Loading {PATH}/GlobalModel_{num}.pth")

        model = load_global_model(f"{PATH}/GlobalModel_{num}.pth")

        prediction_matrix = []
        actual_matrix = []
        acc_matrix = []
        loss_matrix = []

        correct, total = 0, 0
        loss = 0.0
        G += 1

        with torch.no_grad():
            for images, labels in Dataloaders['Test']:

                outputs = model(images)
                batch_loss = criterion(outputs, labels)

                loss += batch_loss.item()

                _, predicted = torch.max(outputs.data, 1)

                total += labels.size(0)
                correct += (predicted == labels).sum().item()

                prediction_matrix.append(predicted.tolist())
                actual_matrix.append(labels.tolist())

        loss /= len(Dataloaders['Test'].dataset)
        accuracy = correct / total

        loss_matrix.append(loss)
        acc_matrix.append(accuracy)

        pred_test[f'Global_{G}'] = prediction_matrix
        actual_test[f'Global_{G}'] = actual_matrix
        accuracy_test[f'Global_{G}'] = acc_matrix
        loss_test[f'Global_{G}'] = loss_matrix

        with open(f'{PATH}/Global_{G}_pred', 'wb') as f:
            pickle.dump(pred_test[f'Global_{G}'], f)

        with open(f'{PATH}/Global_{G}_actual', 'wb') as f:
            pickle.dump(actual_test[f'Global_{G}'], f)

        with open(f'{PATH}/Global_{G}_accuracy', 'wb') as f:
            pickle.dump(accuracy_test[f'Global_{G}'], f)

        with open(f'{PATH}/Global_{G}_loss', 'wb') as f:
            pickle.dump(loss_test[f'Global_{G}'], f)

        print(f"Global {G} → Acc: {accuracy:.4f}, Loss: {loss:.4f}")
    write_round_summary_json(PATH, G)


In [ ]:

# ----------------------------------------
# 0. CONFIG & DIRECTORIES
# ----------------------------------------
DATA_DIR = './data'
CONFUSION_DIR = './confusion'
BATCH_SIZE = 1024  # Process 1024 rows at a time to prevent CUDA OOM
os.makedirs(CONFUSION_DIR, exist_ok=True)

# ----------------------------------------
# LABELS (Attack Classification)
# ----------------------------------------
Label_names = [
        "Normal", "DDoS_UDP", "DDoS_ICMP", "DDoS_TCP", "DDoS_HTTP", "Password",
        "Vulnerability_scanner", "SQL_injection", "Uploading", "Backdoor", "Port_Scanning",
        "XSS", "Ransomware", "MITM", "OS_Fingerprinting"
    ]

Label_numbers = [0, 1, 2, 3, 4, 5, 6, 7,8,9,10,11,12,13,14]

# ----------------------------------------
# Updated Confusion Matrix Saver Function
# ----------------------------------------
def save_confusion_matrix(actual, predicted, label_numbers, label_names, file_name, accuracy):
    label_map = dict(zip(label_numbers, label_names))
    
    # Generate CM for the 8 classes defined
    present_labels = sorted(set(actual) | set(predicted))
    cm = confusion_matrix(actual, predicted, labels=present_labels)
    class_labels = [label_map.get(num, f"Unknown_{num}") for num in present_labels]

    

    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='magma',
                xticklabels=class_labels,
                yticklabels=class_labels)

    plt.xlabel("Predicted Labels", fontsize=14)
    plt.ylabel("Actual Labels", fontsize=14)
    
    # --- ADDED ACCURACY TO TITLE ---
    plt.title(f"Confusion Matrix: {file_name}\nAccuracy: {accuracy:.4f}", fontsize=16)
    
    plt.tight_layout()
    
    # Save to the /confusion folder
    save_path = os.path.join(CONFUSION_DIR, f"{file_name}_cm.png")
    plt.savefig(save_path)
    plt.close() 
    print(f"Saved diagram with accuracy: {save_path}")

def evaluate_csv_datasets(
    data_dir,
    model,
    scaler,
    train_features,
    device,
    batch_size,
    label_numbers,
    label_names,
    output_dir
):

    csv_files = glob.glob(os.path.join(data_dir, "*.csv"))

    label_map = dict(zip(label_numbers, label_names))

    for file_path in csv_files:

        file_name = os.path.basename(file_path).replace(".csv", "")
        print(f"\nProcessing: {file_name}...")

        try:
            df = pd.read_csv(file_path)

            if df.empty:
                print(f"Skip {file_name}: Empty file")
                continue

           
            # Feature alignment
           
            X_new = df.iloc[:, :-1]
            y_true = df.iloc[:, -1].astype(int).values

            X_new = X_new.reindex(columns=train_features, fill_value=0)

           
            # Scaling
           
            X_scaled = scaler.transform(X_new)

           
            # Convert to tensors
           
            X_tensor = torch.tensor(X_scaled).float()
            y_tensor = torch.tensor(y_true).long()

            dataset = TensorDataset(X_tensor, y_tensor)

            loader = DataLoader(
                dataset,
                batch_size=batch_size,
                shuffle=False
            )

           
            # Run inference
           
            avg_loss, acc, preds, actuals, *_ = test(model, loader)

            print(f"\nAccuracy for {file_name}: {acc:.4f}")

           
            # Build full analysis dataframe
           
            comparison_df = pd.DataFrame(X_new)

            comparison_df.insert(0, "Row_Index", range(len(preds)))

            comparison_df["Actual_Label_ID"] = actuals
            comparison_df["Predicted_Label_ID"] = preds

            comparison_df["Actual_Label_Name"] = (
                comparison_df["Actual_Label_ID"].map(label_map)
            )

            comparison_df["Predicted_Label_Name"] = (
                comparison_df["Predicted_Label_ID"].map(label_map)
            )

            comparison_df["Correct"] = (
                comparison_df["Actual_Label_ID"]
                == comparison_df["Predicted_Label_ID"]
            )

           
            # Print preview
           
            print("\nSample Predictions (first 20 rows):")

            preview_cols = [
                "Row_Index",
                "Actual_Label_Name",
                "Predicted_Label_Name",
                "Correct"
            ]

            print(comparison_df[preview_cols].head(20))

           
            # Save analysis CSV
           
            output_csv_path = os.path.join(
                output_dir,
                f"{file_name}_line_analysis.csv"
            )

            comparison_df.to_csv(output_csv_path, index=False)

            print(f"\nSaved analysis CSV to: {output_csv_path}")

           
            # Save confusion matrix
           
            save_confusion_matrix(
                actual=actuals,
                predicted=preds,
                label_numbers=label_numbers,
                label_names=label_names,
                file_name=file_name,
                accuracy=acc
            )

            torch.cuda.empty_cache()

        except Exception as e:

            print(f"Error processing {file_name}: {e}")


# ----------------------------------------
# Initialize model & scaler
# ----------------------------------------

model = load_global_model(f"{PATH}/GlobalModel_{max_num}.pth").to(DEVICE)

with open(f"{PATH}/scaler.pkl", "rb") as f:
    scaler = pickle.load(f)

train_features = scaler.feature_names_in_

print(f"Model loaded on {DEVICE}")

# ----------------------------------------
# Run evaluation
# ----------------------------------------

evaluate_csv_datasets(
    data_dir=DATA_DIR,
    model=model,
    scaler=scaler,
    train_features=train_features,
    device=DEVICE,
    batch_size=BATCH_SIZE,
    label_numbers=Label_numbers,
    label_names=Label_names,
    output_dir=CONFUSION_DIR
)
